# SkinSense AI — Day 2: Full Training, Fine-Tuning, Evaluation
**Member 2 — ML Engineer**

Goal today: replace the Day 1 quick-test model with a properly trained one, and get real accuracy numbers + a confusion matrix to put in the README/demo.

Steps:
1. Re-download + organize dataset (same as Day 1, Colab sessions don't persist)
2. Train in two phases: frozen base (warmup) → fine-tune top MobileNetV2 layers
3. Evaluate on the held-out test set
4. Save final `skin_model.h5` + class index mapping

## 1. Kaggle API + Download + Organize + Split
Same as Day 1 — fill in your username/key below.

In [ ]:
!pip install -q kaggle split-folders

import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_creds = {"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_API_KEY"}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API ready.')

In [ ]:
!mkdir -p /content/raw
%cd /content/raw
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q -o skin-cancer-mnist-ham10000.zip -d ham10000
!kaggle datasets download -d shubhamgoel27/dermnet
!unzip -q -o dermnet.zip -d dermnet
!kaggle datasets download -d ahdasdwdasd/our-normal-skin
!unzip -q -o our-normal-skin.zip -d normal_skin
print('Downloads complete.')

In [ ]:
import os, shutil, glob
import pandas as pd

BASE = '/content/dataset'
CLASSES = ['melanoma', 'nevus', 'basal_cell_carcinoma', 'eczema', 'normal']
for c in CLASSES:
    os.makedirs(f'{BASE}/{c}', exist_ok=True)

meta = pd.read_csv('/content/raw/ham10000/HAM10000_metadata.csv')
ham_img_dirs = glob.glob('/content/raw/ham10000/HAM10000_images_part_*')
dx_to_class = {'mel': 'melanoma', 'nv': 'nevus', 'bcc': 'basal_cell_carcinoma'}

img_lookup = {}
for d in ham_img_dirs:
    for fp in glob.glob(f'{d}/*.jpg'):
        img_id = os.path.splitext(os.path.basename(fp))[0]
        img_lookup[img_id] = fp

copied = {k: 0 for k in dx_to_class.values()}
for _, row in meta.iterrows():
    dx = row['dx']
    if dx in dx_to_class:
        img_id = row['image_id']
        src = img_lookup.get(img_id)
        if src:
            dst_class = dx_to_class[dx]
            shutil.copy(src, f'{BASE}/{dst_class}/{img_id}.jpg')
            copied[dst_class] += 1
print('HAM10000 copied:', copied)

eczema_dirs = glob.glob('/content/raw/dermnet/**/*czema*', recursive=True)
eczema_dirs = [d for d in eczema_dirs if os.path.isdir(d)]
count = 0
for d in eczema_dirs:
    for fp in glob.glob(f'{d}/*'):
        if fp.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(fp, f"{BASE}/eczema/{count}_{os.path.basename(fp)}")
            count += 1
print('Eczema images copied:', count)

normal_files = glob.glob('/content/raw/normal_skin/**/*', recursive=True)
normal_files = [f for f in normal_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
count = 0
for fp in normal_files:
    shutil.copy(fp, f"{BASE}/normal/{count}_{os.path.basename(fp)}")
    count += 1
print('Normal images copied:', count)

for c in CLASSES:
    n = len(glob.glob(f'{BASE}/{c}/*'))
    print(f'{c}: {n} images')

**Check the counts above carefully.** If one class has far fewer images than others (common: eczema or basal_cell_carcinoma), we use `class_weight` in training below to compensate — no need to manually balance folders.

In [ ]:
import splitfolders
splitfolders.ratio('/content/dataset', output='/content/split', seed=42, ratio=(0.8, 0.1, 0.1))
print('Split complete.')

## 2. Data generators + class weights
Class weights tell the model to pay more attention to underrepresented classes, instead of just predicting the majority class every time.

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.85, 1.15],
    width_shift_range=0.1,
    height_shift_range=0.1
).flow_from_directory('/content/split/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    '/content/split/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    '/content/split/test', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

print('Class index mapping (give this to Member 1 — confirm it matches Day 1):')
print(train_gen.class_indices)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights = dict(enumerate(class_weights_arr))
print('Class weights:', class_weights)

## 3. Phase 1 — Warmup training (frozen base)
Train the new classification head first while MobileNetV2's pretrained weights stay frozen. This is more epochs than Day 1's quick test (10 vs 3), with early stopping so it doesn't overfit.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, callbacks

base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_warmup = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop]
)

## 4. Phase 2 — Fine-tuning
Unfreeze the top layers of MobileNetV2 and train with a much smaller learning rate, so the pretrained features adapt slightly to skin images without being destroyed.

In [ ]:
base_model.trainable = True

# Freeze all layers except the last 30 — keeps early generic features intact,
# lets later layers adapt to skin-specific patterns.
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

early_stop_ft = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop_ft]
)

## 5. Evaluate on the test set
This is the number that actually matters — performance on data the model never saw during training or validation.

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

test_gen.reset()
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — SkinSense AI Test Set')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

**Read the classification report carefully.** Look at per-class precision/recall, not just overall accuracy — a class with few images (often eczema or BCC) may have much worse recall than the rest. That's worth mentioning honestly in your README/demo rather than hiding it.

## 6. Save final model + assets

In [ ]:
model.save('/content/skin_model.h5')
print('Saved skin_model.h5')

from google.colab import files
files.download('/content/skin_model.h5')
files.download('/content/confusion_matrix.png')

## Day 2 checklist
- [ ] Trained with warmup + fine-tuning phases
- [ ] Test accuracy recorded: ______
- [ ] Confusion matrix saved and reviewed for weak classes
- [ ] Class index mapping double-checked against Day 1 (should be identical — same alphabetical folder order)
- [ ] `skin_model.h5` (final version) + `confusion_matrix.png` pushed to `model/` folder, replacing Day 1's quick-test file
- [ ] Updated `model/README.md` with: final test accuracy, per-class performance notes, confirmed class index mapping
- [ ] Message sent to Member 1 confirming model is ready for integration